# SQUID Analysis v2 — Instrument, Volume and Demagnetization Corrections

Version 2 of the FunMaP SQUID batch-analysis workflow. It extends
`SQUID_analysis_Caps.ipynb` (v1) with four corrections that v1 does not apply,
following the processing notes from Prof. E. Goering (MPI-FKF, 7 July 2026):

1. **Quantum Design filling-factor correction** — raw EMU is divided by a
   geometry- and VSM-amplitude-dependent factor. v1 applies none.
2. **Absolute magnetization units** — `emu/cm^3`, `A/m` and `T` (as `mu0*M`),
   instead of only normalised `M/Msat` and per-particle EMU.
3. **Demagnetizing-field correction** — converts applied field to internal
   field, `mu0*H_int = mu0*H_app - N * mu0*M`. This is a *different* operation
   from v1's diamagnetic slope subtraction; v2 does both, in the right order.
4. **IP / OOP pairing** — processes two geometries and optionally rescales the
   OOP data to the IP saturation, since the OOP filling-factor correction
   carries the larger error.

v1 is left untouched and remains the validated single-geometry production path.

---

## Important: the reference numbers are not transferable

The constants quoted in the source email
(`qdcorrOOP = 1.228`, `qdcorrIP = 0.965`, `Nc = 0.780216`, `Nab = 0.109892`)
describe a **4.95 mm x 0.56 mm magnetic foil disc**, not FePt caps on SiO2
spheres. They are kept below only as a validation preset for reproducing that
dataset. For FePt cap batches you must supply your own values — see the
demagnetizing-factor section for why there is no single rigorous `N` for a
sphere monolayer.

---

## Correction order

```text
raw m (emu), H (Oe)
  -> 1. divide by QD filling factor          -> instrument-corrected emu
  -> 2. subtract linear diamagnetic slope    -> ferromagnetic emu
  -> 3. divide by FePt volume                -> M in emu/cm^3, A/m, T
  -> 4. shear by demagnetizing factor        -> mu0*H_int
  -> 5. (optional) scale OOP to IP saturation
```

Steps 1 and 2 are both linear, but step 1 comes first so that the reported
background slope is expressed in instrument-corrected units. Step 4 must come
after step 3 because the shear term needs `M` in field units.

In [ ]:
"""
SQUID Measurement Analysis v2 — Instrument, Volume & Demagnetization Corrections
===============================================================================
Loads raw SQUID .dat files for one or two measurement geometries (in-plane and
out-of-plane), applies the Quantum Design filling-factor correction, removes the
diamagnetic substrate background, converts to absolute magnetization units, and
applies a demagnetizing-field (shearing) correction.

Extends SQUID_analysis_Caps.ipynb (v1), which is left unmodified.

Inputs  : Raw SQUID .dat files (Quantum Design MPMS3 format, [Data] header)
Outputs : PNG + SVG comparison figure, .txt report with the full correction
          chain, and averaged CSVs in both applied- and internal-field axes
          — all saved to a timestamped batch folder.

# -----------------------------------------------------------------------------
# Copyright (C) 2026 Max-Planck-Gesellschaft zur Foerderung der Wissenschaften e.V.
#               2026 University of Stuttgart
# Authors: Natalia Gonzalez-Vazquez, Andrew K. Schulz
# SPDX-License-Identifier: GPL-3.0-or-later
#
# This file is part of FunMaP.
# FunMaP is free software: you can redistribute it and/or modify it under
# the terms of the GNU General Public License as published by the Free
# Software Foundation, either version 3 of the License, or (at your option)
# any later version.
#
# FunMaP is distributed in the hope that it will be useful, but WITHOUT ANY
# WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS
# FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License along with
# FunMaP. If not, see <https://www.gnu.org/licenses/>.
# -----------------------------------------------------------------------------
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- GUI ---
import tkinter as tk
from tkinter import filedialog, simpledialog, messagebox

## 1. Settings and constants

`QD_CORR` and `DEMAG_N` are the two entries that must be reviewed for every new
sample geometry. The defaults below are the foil-disc validation preset, *not*
values valid for FePt caps.

In [ ]:
# ===========================================================================
# 1. SETTINGS & CONSTANTS
# ===========================================================================

OUTDIR_BASE   = "squid_batch_analysis_v2"
N_GRID        = 3000    # Points on the common interpolation grid
SAT_FRAC      = 0.05    # Outermost fraction of field used to fit background slope
STD_SMOOTH    = 61      # Moving-average window for SD band smoothing (odd)
DENSITY_G_CM3 = 15.0    # FePt bulk density [g/cm3]
HCP_PACKING   = 0.906   # Hexagonal close-packing fraction for sphere monolayer
OE_TO_T       = 1e-4    # Oersted -> Tesla
MU0           = 4 * np.pi * 1e-7        # Vacuum permeability [T*m/A]
EMU_CM3_TO_AM = 1e3     # 1 emu/cm3 = 1000 A/m

GEOMETRIES = ("IP", "OOP")

# Label used in figure titles and reports. Change it when running a reference
# sample that is not an FePt cap batch (see SAMPLE_VOLUME_CM3 below).
SAMPLE_LABEL = "FePt"

# --- Quantum Design filling-factor corrections -----------------------------
# Raw EMU is DIVIDED by these. They depend on sample shape AND on the VSM
# oscillation amplitude, so they must be recomputed with the QD calculator for
# every new sample geometry.
#
# Preset below: cylinder, 0.56 mm height, 4.95 mm diameter, 1 mm VSM amplitude
# (E. Goering, 7 July 2026) — the magnetic foil disc, NOT the FePt caps.
QD_CORR = {
    "IP":  0.965,
    "OOP": 1.228,
}

# --- Demagnetizing factors -------------------------------------------------
# N along the field direction for each geometry. See section 2 for helpers and
# for why a sphere monolayer of hemispherical caps has no single rigorous N.
#
# Preset below: Moskowitz & Della Torre (1966) values for the foil disc,
# Nc along the short (OOP) axis, Nab in-plane, satisfying Nc + 2*Nab = 1.
DEMAG_N = {
    "IP":  0.109892,
    "OOP": 0.780216,
}

# Set False to skip the shearing correction entirely and report applied field only.
APPLY_DEMAG = True

# --- Magnetic volume -------------------------------------------------------
# None  -> compute the FePt volume from the cap geometry (HCP sphere monolayer),
#          which is the normal FunMaP path.
# float -> use this volume [cm3] directly. Needed for reference samples that are
#          not sphere monolayers, e.g. the foil disc used to validate this
#          notebook: 0.056 * pi * (0.495/2)**2 = 0.010776762 cm3.
SAMPLE_VOLUME_CM3 = None

# Scale OOP magnetization to the IP saturation. Recommended by Goering because
# the OOP filling-factor correction carries the larger error.
SCALE_OOP_TO_IP = True

# Keep only the closed cycle, discarding a leading virgin curve and any
# trailing park-at-zero-field segment. Both inject spurious Hc / Mr candidates
# near the origin. See trim_to_closed_cycle for the details.
TRIM_TO_CLOSED_CYCLE = True

## 2. Demagnetizing factors

The shearing correction is

```text
mu0*H_int = mu0*H_app - N * mu0*M
```

Both terms carry a factor `mu0`, so working directly in Tesla avoids any
cgs/SI conversion in the correction itself.

### Choosing N

`demag_oblate_spheroid` is exact for an ellipsoid. A **cylinder is not an
ellipsoid**, and the difference is not negligible: for the 4.95 x 0.56 mm foil
disc the helpers below give

| Method | N along short axis |
|---|---|
| Oblate spheroid (exact, analytic) | 0.845 |
| Cylinder aspect-ratio approximation | 0.815 |
| Moskowitz & Della Torre 1966 (Goering) | 0.780 |

an ~8% spread that propagates directly into the corrected `mu0*M`. Prefer
tabulated cylinder values when the sample really is a cylinder.

### For FePt caps there is no single correct N

A monolayer of hemispherical 60 nm caps on SiO2 spheres is not an ellipsoid,
a cylinder, or a continuous film. Defensible modelling choices are:

- `thin_film` — `N_perp = 1`, `N_para = 0`. The local-film limit; the most
  defensible default, since the cap is 60 nm thick over a micron-scale sphere.
- `sphere` — `N = 1/3` in every direction, treating each capped sphere as a
  compact particle.

Whichever is used must be stated explicitly in the manuscript. The helpers
below print the alternatives so the sensitivity is visible.

In [ ]:
# ===========================================================================
# 2. DEMAGNETIZING-FACTOR HELPERS
# ===========================================================================

def demag_oblate_spheroid(c_over_a):
    """
    Exact demagnetizing factor along the short (c) axis of an oblate spheroid
    with semi-axes a = b > c. Returns 1/3 for a sphere, -> 1 for a flat disc.
    """
    m = float(c_over_a)
    if m >= 1.0:
        return 1.0 / 3.0
    e = np.sqrt(1.0 - m**2)
    return (1.0 / e**2) * (1.0 - (m / e) * np.arcsin(e))


def demag_cylinder_axial(height, diameter):
    """
    Aspect-ratio approximation for a cylinder magnetized along its axis.
    Exact in both limits (N -> 1 for a flat disc, N -> 0 for a long rod) and
    returns 1/3 when height == diameter, but it is an interpolation, not a
    rigorous result. Prefer tabulated values (e.g. Moskowitz & Della Torre 1966).
    """
    return 1.0 / (1.0 + 2.0 * float(height) / float(diameter))


def demag_triplet(n_axial):
    """
    Given the axial factor, return (N_axial, N_transverse) enforcing the
    trace rule N_axial + 2*N_transverse = 1 for a body of revolution.
    """
    n_axial = float(n_axial)
    return n_axial, (1.0 - n_axial) / 2.0


def check_demag_trace(n_axial, n_transverse, tol=1e-6):
    """Warn if the supplied demagnetizing factors violate N_c + 2*N_ab = 1."""
    total = n_axial + 2.0 * n_transverse
    ok = abs(total - 1.0) < tol
    if not ok:
        print(f"[!] Demagnetizing factors violate the trace rule: "
              f"N_c + 2*N_ab = {total:.6f}, expected 1.000000")
    return ok


def report_demag_options(height_mm, diameter_mm):
    """Print the available N estimates for a disc-shaped sample, for comparison."""
    n_sph = demag_oblate_spheroid((height_mm / 2.0) / (diameter_mm / 2.0))
    n_cyl = demag_cylinder_axial(height_mm, diameter_mm)
    print(f"  Demagnetizing factor along the short axis "
          f"({diameter_mm} mm dia x {height_mm} mm):")
    print(f"    oblate spheroid (exact)      : {n_sph:.6f}")
    print(f"    cylinder approximation       : {n_cyl:.6f}")
    print(f"    configured (DEMAG_N['OOP'])  : {DEMAG_N['OOP']:.6f}")
    print(f"    thin-film limit              : 1.000000")
    return n_sph, n_cyl

## 3. Loading and correction chain

In [ ]:
# ===========================================================================
# 3. LOADING & CORRECTION CHAIN
# ===========================================================================

def get_fept_volume_and_mass(substrate_area_mm2, sphere_diam_um, thick_nm):
    """
    Estimate total FePt volume and mass for an HCP monolayer of spheres.
    Returns: total_vol_cm3, mass_g, vol_per_cap_cm3, n_spheres
    Unchanged from v1 so that v1 and v2 volumes stay directly comparable.
    """
    sphere_r_mm = (sphere_diam_um / 2) * 1e-3
    n_spheres   = (substrate_area_mm2 * HCP_PACKING) / (2 * np.sqrt(3) * sphere_r_mm**2)
    R_cm        = (sphere_diam_um / 2) * 1e-4
    t_cm        = thick_nm * 1e-7
    vol_cap_cm3 = (2/3) * np.pi * ((R_cm + t_cm)**3 - R_cm**3)
    total_vol   = n_spheres * vol_cap_cm3
    return total_vol, total_vol * DENSITY_G_CM3, vol_cap_cm3, n_spheres


def read_squid_dat(path):
    """
    Read a Quantum Design .dat file. Returns a DataFrame of the [Data] block,
    or None if no [Data] header is present.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    start_idx = next((i + 1 for i, l in enumerate(lines) if "[Data]" in l), -1)
    if start_idx == -1:
        return None
    return pd.read_csv(path, skiprows=start_idx).dropna(
        subset=["Magnetic Field (Oe)", "Moment (emu)"]
    )


def estimate_sweep_rate(df):
    """
    Estimate the field sweep rate in Oe/s from a time column, if one is present.
    Returns None when the file carries no timestamps (e.g. the synthetic
    sample_data files). Goering used 10 Oe/s for IP and 100 Oe/s for OOP, and
    found a full-speed 700 Oe/s sweep to differ only slightly — so this is
    recorded as metadata rather than used to correct anything.
    """
    tcol = next((c for c in ("Time Stamp (sec)", "Time Stamp (s)", "Time (s)")
                 if c in df.columns), None)
    if tcol is None:
        return None
    t = df[tcol].values.astype(float)
    h = df["Magnetic Field (Oe)"].values.astype(float)
    dt = np.diff(t)
    good = dt > 0
    if good.sum() < 3:
        return None
    return float(np.median(np.abs(np.diff(h)[good] / dt[good])))


def subtract_linear_background(h, m):
    """
    Remove linear diamagnetic/paramagnetic substrate background.

    Fits the positive tail (h > 95% of h_max) and negative tail
    (h < -95% of h_max) independently, then averages the two slopes
    for robustness before subtracting. Carried over from v1.

    NOTE: this is a *substrate/holder* correction. It is unrelated to the
    demagnetizing correction, which rescales the field axis rather than the
    moment axis. v2 applies both.
    """
    h, m      = np.asarray(h), np.asarray(m)
    h_abs_max = np.max(np.abs(h))
    mask_pos  = h >  h_abs_max * (1 - SAT_FRAC)
    mask_neg  = h < -h_abs_max * (1 - SAT_FRAC)
    slope_pos, _ = np.polyfit(h[mask_pos], m[mask_pos], 1) if mask_pos.sum() > 2 else (0, 0)
    slope_neg, _ = np.polyfit(h[mask_neg], m[mask_neg], 1) if mask_neg.sum() > 2 else (0, 0)
    slope = (slope_pos + slope_neg) / 2
    return m - slope * h, slope


def to_magnetization(m_emu, vol_cm3):
    """
    Convert a moment in EMU to magnetization in emu/cm3, A/m and Tesla (mu0*M).
    """
    M_emu_cm3 = m_emu / vol_cm3
    M_A_m     = M_emu_cm3 * EMU_CM3_TO_AM
    M_T       = MU0 * M_A_m
    return M_emu_cm3, M_A_m, M_T


def max_valid_demag_factor(mu0_h_app_T, mu0_M_T, n_slope_grid=400):
    """
    Largest demagnetizing factor for which the shearing correction stays
    single-valued: N_crit = 1 / max|d(mu0*M)/d(mu0*H)|.

    Beyond N_crit the sheared field axis folds back on itself and the corrected
    loop becomes multivalued, so the correction is no longer meaningful.

    The slope is evaluated on the descending branch resampled onto a uniform
    field grid. Raw point-to-point differencing must NOT be used here: SQUID
    sweeps repeat or barely advance the field near the turning points, so a
    finite difference there is dominated by measurement noise and understates
    N_crit by more than an order of magnitude.
    """
    h = np.asarray(mu0_h_app_T, dtype=float)
    M = np.asarray(mu0_M_T, dtype=float)
    if h.size < 4:
        return np.inf

    # Isolate the descending branch so the field axis is single-valued
    i_max, i_min = np.argmax(h), np.argmin(h)
    a, b = (i_max, i_min) if i_max < i_min else (i_min, i_max)
    hb, Mb = h[a:b+1], M[a:b+1]
    if hb.size < 4 or np.ptp(hb) == 0:
        return np.inf

    order = np.argsort(hb)
    grid = np.linspace(hb.min(), hb.max(), n_slope_grid)
    M_grid = np.interp(grid, hb[order], Mb[order])
    slope = np.max(np.abs(np.diff(M_grid) / np.diff(grid)))
    return (1.0 / slope) if slope > 0 else np.inf


def demag_correct_field(mu0_h_app_T, mu0_M_T, n_demag):
    """
    Shear the field axis: mu0*H_int = mu0*H_app - N * mu0*M.

    Both terms are already in Tesla, so no cgs/SI factor enters here.

    Warns when N exceeds N_crit, i.e. when the shear makes the internal field
    non-monotonic. This is a hard limit, not a cosmetic one: a hard magnet with
    a steep switching region admits only a modest shear before the corrected
    loop becomes multivalued. For FePt (mu0*Ms ~ 1.26 T) the thin-film limit
    N = 1 will typically violate it, so check the printed N_crit before trusting
    a large-N correction.
    """
    mu0_h_app_T = np.asarray(mu0_h_app_T)
    mu0_M_T     = np.asarray(mu0_M_T)
    n_crit      = max_valid_demag_factor(mu0_h_app_T, mu0_M_T)

    if n_demag > n_crit:
        print(f"    [!] N = {n_demag:.4f} exceeds N_crit = {n_crit:.4f} — the "
              f"sheared loop is multivalued and the correction is not valid here.")
        print(f"        Reduce N, or report applied field only (APPLY_DEMAG = False).")
    elif n_demag > 0.7 * n_crit:
        print(f"    [i] N = {n_demag:.4f} is within 30% of N_crit = {n_crit:.4f} — "
              f"the shear is near its validity limit.")
    return mu0_h_app_T - n_demag * mu0_M_T

## 4. Branch handling and metrics

Note on coercivity: the shearing correction vanishes at `M = 0`, so **`Hc` is
identical in applied and internal field**. Remanence is *not* — at
`mu0*H_app = 0` the internal field is `-N * mu0*Mr`, so `Mr` is quoted at zero
applied field by convention and that convention is recorded in the report.

In [ ]:
# ===========================================================================
# 4. BRANCH HANDLING & METRICS
# ===========================================================================

def trim_to_closed_cycle(h, m, sat_frac=0.95):
    """
    Keep only the closed hysteresis cycle: everything between the FIRST and the
    LAST time the field reaches saturation. Returns (h, m, n_head, n_tail).

    This removes two artefacts that real MPMS sequences routinely produce:

    * the leading virgin curve, when the sweep starts from a demagnetized state
      at H = 0. It passes through the origin, contributing a spurious M = 0
      crossing near H = 0 and a spurious H = 0 crossing near M = 0, which drags
      both Hc and Mr toward zero.
    * a trailing "park at zero field" segment. Left in place, `split_branches`
      wraps the array across that discontinuity and the resulting seam looks
      like an extra H = 0 crossing at M ~ 0.

    The cut indices must come from the first/last approach to saturation, NOT
    from argmax(h): a symmetric loop ends at the field it started cycling from,
    so floating-point noise routinely puts the global maximum on the last point,
    which would discard an entire branch.
    """
    h, m = np.asarray(h), np.asarray(m)
    h_abs_max = np.max(np.abs(h))
    if h.size < 3 or h_abs_max == 0:
        return h, m, 0, 0
    at_sat = np.abs(h) >= sat_frac * h_abs_max
    if not np.any(at_sat):
        return h, m, 0, 0
    idx = np.where(at_sat)[0]
    start, end = int(idx[0]), int(idx[-1])
    if end - start < 3:
        return h, m, 0, 0
    return h[start:end + 1], m[start:end + 1], start, h.size - 1 - end


def split_branches(h, m):
    """
    Split a hysteresis loop into descending (H+ -> H-) and ascending (H- -> H+)
    branches. Carried over from v1.
    """
    idx_max, idx_min = np.argmax(h), np.argmin(h)
    if idx_max < idx_min:
        h_desc, m_desc = h[idx_max:idx_min+1], m[idx_max:idx_min+1]
        h_asc  = np.concatenate([h[idx_min:], h[:idx_max]])
        m_asc  = np.concatenate([m[idx_min:], m[:idx_max]])
    else:
        h_asc,  m_asc  = h[idx_min:idx_max+1], m[idx_min:idx_max+1]
        h_desc = np.concatenate([h[idx_max:], h[:idx_min]])
        m_desc = np.concatenate([m[idx_max:], m[:idx_min]])
    return (h_desc, m_desc), (h_asc, m_asc)


def moving_average(y, n):
    """Box-car moving average of window n (forced odd)."""
    if n < 3:
        return y
    if n % 2 == 0:
        n += 1
    return np.convolve(np.asarray(y), np.ones(n) / n, mode="same")


def interp_to_grid(grid, x, y):
    """Interpolate (x, y) onto grid, sorting by x first."""
    order = np.argsort(x)
    return np.interp(grid, np.asarray(x)[order], np.asarray(y)[order])


def zero_crossings(x, y):
    """
    x-values at which y crosses zero, linearly interpolated.
    Returns every crossing; callers take the median so that noise around
    zero does not pick an arbitrary single point.
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    out = list(x[y == 0.0])
    idx = np.where(y[:-1] * y[1:] < 0)[0]
    for i in idx:
        t = y[i] / (y[i] - y[i + 1])
        out.append(x[i] + t * (x[i + 1] - x[i]))
    return np.asarray(out, dtype=float)


def points_near_zero_field(mu0_h_T, frac=0.02):
    """
    Number of samples within `frac` of full scale of zero field. Remanence is
    read at H = 0, so a fast sweep that places only a handful of points there
    gives an unreliable Mr even though Hc and Msat remain sound.
    """
    h = np.asarray(mu0_h_T, dtype=float)
    scale = np.max(np.abs(h))
    return int(np.sum(np.abs(h) < frac * scale)) if scale else 0


def get_metrics(mu0_h_T, M_A_m, n_grid=2000):
    """
    Extract coercivity [T], squareness (Mr/Ms) and hysteresis loss [J/kg]
    directly in SI, from field in Tesla and magnetization in A/m.

    Hc and Mr are evaluated BRANCH-WISE:
      Hc = mean over branches of |field where M crosses zero|
      Mr = mean over branches of |M where the field crosses zero|

    Taking `argmin(|M|)` over the whole loop instead — as v1 does — silently
    returns the wrong point whenever the sweep starts near H = 0 rather than at
    saturation, because the virgin-curve origin is then the closest point to
    M = 0. Real MPMS sweeps routinely start at zero field, so this matters.

    W_hyst is the area enclosed between the descending and ascending branches,
    integral of (M_desc - M_asc) d(mu0*H), which is well defined even when the
    recorded path does not close exactly. Units: A/m * T = J/m3, then / density.
    """
    (h_d, m_d), (h_a, m_a) = split_branches(np.asarray(mu0_h_T, dtype=float),
                                            np.asarray(M_A_m, dtype=float))

    hc_vals, mr_vals = [], []
    for h_b, m_b in ((h_d, m_d), (h_a, m_a)):
        if h_b.size < 2:
            continue
        c = zero_crossings(h_b, m_b)          # field where M = 0  -> Hc
        if c.size:
            hc_vals.append(np.median(np.abs(c)))
        r = zero_crossings(m_b, h_b)          # M where H = 0      -> Mr
        if r.size:
            mr_vals.append(np.median(np.abs(r)))

    hc_t = float(np.mean(hc_vals)) if hc_vals else np.nan
    mr   = float(np.mean(mr_vals)) if mr_vals else np.nan
    msat = float(np.max(np.abs(M_A_m)))
    mr_ms = (mr / msat) if (msat and not np.isnan(mr)) else np.nan

    # Enclosed loop area over the field range common to both branches
    w_j_m3 = np.nan
    if h_d.size > 1 and h_a.size > 1:
        lo = max(h_d.min(), h_a.min())
        hi = min(h_d.max(), h_a.max())
        if hi > lo:
            g = np.linspace(lo, hi, n_grid)
            w_j_m3 = abs(np.trapezoid(interp_to_grid(g, h_d, m_d)
                                      - interp_to_grid(g, h_a, m_a), g))
    w_j_kg = w_j_m3 / (DENSITY_G_CM3 * 1e3)
    return hc_t, mr_ms, msat, w_j_m3, w_j_kg


def anisotropy_from_pair(mu0_h_easy, M_easy, mu0_h_hard, M_hard, msat):
    """
    Approximate effective anisotropy from a demag-corrected easy/hard axis pair.

    K_eff = integral from 0 to Ms of (mu0*H_hard - mu0*H_easy) dM   [J/m3]
    mu0*H_K is reported as the hard-axis field reaching 95% of saturation.

    APPROXIMATE. For curved FePt caps the easy axis is spatially distributed, so
    this is an ensemble-effective number, not a single-crystal anisotropy.
    Returns (K_eff_J_m3, mu0_HK_T) with NaN where it cannot be evaluated.
    """
    try:
        m_axis = np.linspace(0.0, 0.95 * msat, 500)
        h_e = interp_to_grid(m_axis, M_easy,  mu0_h_easy)
        h_h = interp_to_grid(m_axis, M_hard,  mu0_h_hard)
        k_eff = abs(np.trapezoid(h_h - h_e, m_axis))
        idx = np.argmin(np.abs(M_hard - 0.95 * msat))
        return k_eff, abs(mu0_h_hard[idx])
    except Exception as e:
        print(f"    [!] Anisotropy estimate failed: {e}")
        return np.nan, np.nan

## 5. Per-geometry batch processing

Runs the full correction chain over every file of one geometry and returns
both the individual curves (for plotting) and the branch-averaged mean +/- SD.

In [ ]:
# ===========================================================================
# 5. PER-GEOMETRY BATCH PROCESSING
# ===========================================================================

def process_geometry(paths, geom, vol_cm3, do_bkg, h_grid):
    """
    Apply the full v2 correction chain to every file of one geometry.

    Returns a dict with individual curves, branch-averaged means and SDs on the
    applied-field grid, per-file metrics, and the correction metadata.
    """
    qd = QD_CORR[geom]
    n_demag = DEMAG_N[geom] if APPLY_DEMAG else 0.0

    print(f"\n--- {geom} ---")
    print(f"  QD filling factor : {qd}   (raw EMU divided by this)")
    print(f"  Demagnetizing N   : {n_demag}"
          f"{'' if APPLY_DEMAG else '   (demag correction disabled)'}")

    out = {
        "geom": geom, "qd": qd, "n_demag": n_demag,
        "h_app": [], "M_T": [], "h_int": [], "M_A_m": [], "m_emu": [],
        "desc": [], "asc": [], "metrics": [], "slopes": [], "rates": [],
        "names": [], "n_crit": [],
    }

    for p in paths:
        name = os.path.basename(p)
        try:
            df = read_squid_dat(p)
            if df is None:
                print(f"  [!] No [Data] block in {name} — skipping.")
                continue

            h_oe = df["Magnetic Field (Oe)"].values.astype(float)
            m_raw = df["Moment (emu)"].values.astype(float)

            rate = estimate_sweep_rate(df)
            out["rates"].append(rate)

            if TRIM_TO_CLOSED_CYCLE:
                h_oe, m_raw, n_head, n_tail = trim_to_closed_cycle(h_oe, m_raw)
                if n_head or n_tail:
                    print(f"  [{name}] trimmed to closed cycle: "
                          f"{n_head} leading (virgin curve), "
                          f"{n_tail} trailing (park at H=0)")

            # --- Step 1: Quantum Design filling-factor correction ---
            m_inst = m_raw / qd

            # --- Step 2: diamagnetic substrate background ---
            if do_bkg:
                m_ferro, slope = subtract_linear_background(h_oe, m_inst)
                print(f"  [{name}] slope removed: {slope:.4e} emu/Oe"
                      f"{f' | sweep {rate:.1f} Oe/s' if rate else ''}")
            else:
                m_ferro, slope = m_inst.copy(), 0.0
            out["slopes"].append(slope)

            # --- Step 3: absolute magnetization ---
            _, M_A_m, M_T = to_magnetization(m_ferro, vol_cm3)

            # --- Step 4: demagnetizing shear ---
            mu0_h_app = h_oe * OE_TO_T
            mu0_h_int = (demag_correct_field(mu0_h_app, M_T, n_demag)
                         if APPLY_DEMAG else mu0_h_app.copy())

            out["h_app"].append(mu0_h_app)
            out["h_int"].append(mu0_h_int)
            out["M_T"].append(M_T)
            out["M_A_m"].append(M_A_m)
            out["m_emu"].append(m_ferro)
            out["n_crit"].append(max_valid_demag_factor(mu0_h_app, M_T))
            out["names"].append(name)

            (hd, md), (ha, ma) = split_branches(mu0_h_app, M_T)
            out["desc"].append(interp_to_grid(h_grid, hd, md))
            out["asc"].append(interp_to_grid(h_grid, ha, ma))

            hc, mr_ms, msat, w_m3, w_kg = get_metrics(mu0_h_app, M_A_m)
            n_near = points_near_zero_field(mu0_h_app)
            if n_near < 10:
                print(f"    [i] only {n_near} points near zero field — "
                      f"Mr/Ms is poorly constrained for this sweep "
                      f"(Hc and Msat are unaffected).")
            out["metrics"].append({
                "name": name, "geometry": geom,
                "Hc_T": hc, "Mr_over_Ms": mr_ms,
                "Msat_A_per_m": msat, "mu0_Msat_T": MU0 * msat,
                "Whyst_J_per_m3": w_m3, "Whyst_J_per_kg": w_kg,
                "sweep_rate_Oe_per_s": rate,
                "n_points_near_H0": n_near,
            })

        except Exception as e:
            print(f"  [!] Error in {name}: {e}")

    if not out["desc"]:
        return None

    out["desc_avg"], out["desc_sd"] = np.mean(out["desc"], axis=0), np.std(out["desc"], axis=0)
    out["asc_avg"],  out["asc_sd"]  = np.mean(out["asc"],  axis=0), np.std(out["asc"],  axis=0)
    out["msat_T"] = float(np.max(np.abs(np.concatenate([out["desc_avg"], out["asc_avg"]]))))
    return out

## 6. Main

Prompts for geometry, sample dimensions and files, runs both geometries
through the chain, and writes the figure, report and CSVs.

In [ ]:
# ===========================================================================
# 6. MAIN
# ===========================================================================

def main():

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    diam  = simpledialog.askfloat("Input", "Sphere diameter (um):", initialvalue=3.0)
    thick = simpledialog.askfloat("Input", "Cap thickness (nm):",   initialvalue=60.0)
    area  = simpledialog.askfloat("Input", "Substrate area (mm2):", initialvalue=25.0)

    if any(v is None for v in [diam, thick, area]):
        messagebox.showwarning("Warning", "All geometry inputs are required.")
        root.destroy()
        return

    paths_by_geom = {}
    for geom in GEOMETRIES:
        sel = filedialog.askopenfilenames(
            title=f"Select {geom} SQUID .dat files for the {diam} um batch "
                  f"(Cancel to skip {geom})")
        if sel:
            paths_by_geom[geom] = sel
    root.destroy()

    if not paths_by_geom:
        print("[!] No files selected.")
        return

    do_bkg = input("Apply diamagnetic background subtraction? (yes/no): ").strip().lower() in ("yes", "y")

    batch_folder = os.path.join(OUTDIR_BASE, f"Batch_v2_{diam}um_{ts}")
    os.makedirs(batch_folder, exist_ok=True)

    vol_cm3, mass_g, vol_cap_cm3, n_spheres = get_fept_volume_and_mass(area, diam, thick)
    if SAMPLE_VOLUME_CM3 is not None:
        vol_cm3 = float(SAMPLE_VOLUME_CM3)
        mass_g  = vol_cm3 * DENSITY_G_CM3
        print(f"\n[i] Using SAMPLE_VOLUME_CM3 override: {vol_cm3:.9f} cm3 "
              f"(cap-geometry volume ignored)")
    print(f"\nMagnetic mass: {mass_g:.3e} g | Volume: {vol_cm3:.3e} cm3 | "
          f"N particles: {n_spheres:.3e}")

    # Trace-rule sanity check on the configured demagnetizing factors
    check_demag_trace(DEMAG_N["OOP"], DEMAG_N["IP"])

    h_grid = None
    for sel in paths_by_geom.values():
        df = read_squid_dat(sel[0])
        if df is not None:
            h_T = df["Magnetic Field (Oe)"].values.astype(float) * OE_TO_T
            h_grid = np.linspace(h_T.min(), h_T.max(), N_GRID)
            break
    if h_grid is None:
        print("[!] Could not establish a field grid from the selected files.")
        return

    results = {}
    for geom, sel in paths_by_geom.items():
        r = process_geometry(sel, geom, vol_cm3, do_bkg, h_grid)
        if r:
            results[geom] = r

    if not results:
        print("[!] No files were successfully processed.")
        return

    # --- Step 5: scale OOP to the IP saturation --------------------------
    oop_scale = 1.0
    if SCALE_OOP_TO_IP and {"IP", "OOP"} <= set(results):
        oop_scale = results["IP"]["msat_T"] / results["OOP"]["msat_T"]
        print(f"\nOOP -> IP saturation scale factor: {oop_scale:.4f}")
        print("  (applied because the OOP filling-factor correction carries "
              "the larger error)")

    # ===========================================================================
    # 7. PLOTTING
    # ===========================================================================
    style = {"IP": {"color": "tab:blue"}, "OOP": {"color": "tab:red"}}
    sm = lambda x: moving_average(x, STD_SMOOTH)

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    (ax1, ax2), (ax3, ax4) = axes
    fig.suptitle(
        f"{SAMPLE_LABEL} M(H) v2 — {diam} um batch — "
        f"instrument, volume and demag corrected\n{ts}",
        fontweight="bold")

    # Panel 1 — corrected moment in EMU
    for geom, r in results.items():
        for i, (h, m) in enumerate(zip(r["h_app"], r["m_emu"])):
            ax1.plot(h, m, lw=1.0, alpha=0.7, color=style[geom]["color"],
                     label=f"{geom}  (/{r['qd']})" if i == 0 else None)
    ax1.set_ylabel(r"$m$ (emu)")
    ax1.set_title("Corrected moment (QD filling factor applied)")

    # Panel 2 — absolute magnetization vs APPLIED field
    for geom, r in results.items():
        ax2.fill_between(h_grid, r["desc_avg"] - sm(r["desc_sd"]),
                         r["desc_avg"] + sm(r["desc_sd"]),
                         color=style[geom]["color"], alpha=0.15)
        ax2.fill_between(h_grid, r["asc_avg"] - sm(r["asc_sd"]),
                         r["asc_avg"] + sm(r["asc_sd"]),
                         color=style[geom]["color"], alpha=0.15)
        ax2.plot(h_grid, r["desc_avg"], lw=2, color=style[geom]["color"],
                 label=f"{geom} mean $\\pm$ 1 SD")
        ax2.plot(h_grid, r["asc_avg"], lw=2, color=style[geom]["color"])
    ax2.set_ylabel(r"$\mu_0 M$ (T)")
    ax2.set_title("Absolute magnetization vs applied field")

    # Panel 3 — demag-corrected: internal field
    for geom, r in results.items():
        for i, (h, M) in enumerate(zip(r["h_int"], r["M_T"])):
            ax3.plot(h, M, lw=1.0, alpha=0.7, color=style[geom]["color"],
                     label=f"{geom}  (N = {r['n_demag']:.4f})" if i == 0 else None)
    ax3.set_ylabel(r"$\mu_0 M$ (T)")
    ax3.set_xlabel(r"$\mu_0 H_{\mathrm{int}}$ (T)")
    ax3.set_title("Demagnetization-corrected (internal field)")

    # Panel 4 — OOP rescaled to IP saturation
    for geom, r in results.items():
        s = oop_scale if geom == "OOP" else 1.0
        lbl = f"{geom}" + (f" $\\times$ {s:.3f}" if s != 1.0 else "")
        ax4.plot(h_grid, r["desc_avg"] * s, lw=2, color=style[geom]["color"], label=lbl)
        ax4.plot(h_grid, r["asc_avg"] * s, lw=2, color=style[geom]["color"])
    ax4.set_ylabel(r"$\mu_0 M$ (T)")
    ax4.set_xlabel(r"$\mu_0 H_{\mathrm{app}}$ (T)")
    ax4.set_title("OOP rescaled to IP saturation" if oop_scale != 1.0
                  else "Averaged loops")

    for ax in (ax1, ax2, ax3, ax4):
        ax.axhline(0, color="black", lw=0.5)
        ax.axvline(0, color="black", lw=0.5)
        ax.grid(True, ls=":", alpha=0.6)
        ax.legend(loc="lower right", fontsize=8)
    ax1.set_xlabel(r"$\mu_0 H_{\mathrm{app}}$ (T)")
    ax2.set_xlabel(r"$\mu_0 H_{\mathrm{app}}$ (T)")

    plt.tight_layout()
    base = os.path.join(batch_folder, f"SQUID_v2_{diam}um_{ts}")
    fig.savefig(base + ".png", dpi=300)
    fig.savefig(base + ".svg")
    plt.close(fig)
    print("\nPlot saved.")

    # ===========================================================================
    # 8. REPORT & EXPORTS
    # ===========================================================================
    all_metrics = [m for r in results.values() for m in r["metrics"]]
    df_res = pd.DataFrame(all_metrics)

    k_eff, mu0_hk = np.nan, np.nan
    if {"IP", "OOP"} <= set(results):
        ip, oop = results["IP"], results["OOP"]
        msat = max(np.max(np.abs(ip["M_A_m"][0])), np.max(np.abs(oop["M_A_m"][0])))
        k_eff, mu0_hk = anisotropy_from_pair(
            ip["h_int"][0], ip["M_A_m"][0],
            oop["h_int"][0], oop["M_A_m"][0], msat)

    report_path = os.path.join(batch_folder, f"Report_v2_{diam}um_{ts}.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("SQUID Analysis Report - v2 (instrument, volume, demagnetization)\n")
        f.write("=" * 70 + "\n")
        f.write(f"Timestamp            : {ts}\n")
        f.write(f"Geometries processed : {', '.join(results)}\n")
        f.write(f"Background corr      : {'Yes' if do_bkg else 'No'}\n\n")

        f.write("Correction chain\n----------------\n")
        f.write("1. raw EMU divided by the QD filling factor\n")
        for geom, r in results.items():
            f.write(f"     {geom:<4}: {r['qd']}\n")
        f.write("2. linear diamagnetic slope subtracted from the saturated tails\n")
        for geom, r in results.items():
            slopes = r["slopes"]
            f.write(f"     {geom:<4}: mean slope {np.mean(slopes):.6e} emu/Oe\n")
        f.write(f"3. divided by magnetic volume {vol_cm3:.6e} cm3 "
                f"-> emu/cm3, A/m, T\n")
        f.write(f"4. demagnetizing shear  mu0*H_int = mu0*H_app - N * mu0*M "
                f"({'applied' if APPLY_DEMAG else 'DISABLED'})\n")
        for geom, r in results.items():
            n_crit = float(np.min(r["n_crit"])) if r["n_crit"] else np.inf
            flag = "  <-- EXCEEDED, sheared loop is multivalued" \
                   if r["n_demag"] > n_crit else ""
            f.write(f"     {geom:<4}: N = {r['n_demag']:.6f}   "
                    f"N_crit = {n_crit:.6f}{flag}\n")
        f.write("     N_crit = 1 / max|d(mu0*M)/d(mu0*H)| is the largest shear that\n")
        f.write("     keeps the corrected loop single-valued.\n")
        f.write(f"5. OOP scaled to IP saturation: "
                f"{'x %.4f' % oop_scale if oop_scale != 1.0 else 'not applied'}\n\n")

        f.write("Sample geometry\n---------------\n")
        f.write(f"Sphere diameter      : {diam} um\n")
        f.write(f"Cap thickness        : {thick} nm\n")
        f.write(f"Substrate area       : {area} mm2\n")
        f.write(f"HCP packing          : {HCP_PACKING}\n")
        f.write(f"N particles (est.)   : {n_spheres:.4e}\n")
        f.write(f"Magnetic mass        : {mass_g:.6e} g\n")
        f.write(f"Magnetic volume      : {vol_cm3:.6e} cm3\n")
        f.write(f"Vol per cap          : {vol_cap_cm3:.4e} cm3\n\n")

        f.write("Conventions\n-----------\n")
        f.write("Hc is identical in applied and internal field (the shear term\n")
        f.write("vanishes at M = 0). Mr/Ms is quoted at zero APPLIED field.\n\n")

        f.write("Per-file metrics\n----------------\n")
        f.write(df_res.to_string(index=False))

        f.write("\n\nPer-geometry summary (mean +/- std)\n")
        for geom in results:
            sub = df_res[df_res["geometry"] == geom]
            f.write(f"\n  {geom}:\n")
            for col in ["Hc_T", "Mr_over_Ms", "mu0_Msat_T", "Whyst_J_per_kg"]:
                f.write(f"    {col:<20}: {sub[col].mean():.6f} +/- {sub[col].std():.6f}\n")

        if not np.isnan(k_eff):
            f.write("\nAnisotropy estimate (APPROXIMATE)\n")
            f.write(f"  K_eff   : {k_eff:.4e} J/m3\n")
            f.write(f"  mu0*H_K : {mu0_hk:.4f} T\n")
            f.write("  Ensemble-effective only - the easy axis of a curved cap\n")
            f.write("  is spatially distributed, so this is not a single-crystal value.\n")

        f.write("\nNOTE: the QD filling factors and demagnetizing factors above are\n")
        f.write("configuration values. Confirm they were computed for THIS sample\n")
        f.write("geometry and VSM amplitude before publishing absolute magnetizations.\n")

    # --- CSV exports -------------------------------------------------------
    export = {"Field_T": h_grid}
    for geom, r in results.items():
        s = oop_scale if geom == "OOP" else 1.0
        export[f"mu0M_{geom}_desc_T"] = r["desc_avg"]
        export[f"mu0M_{geom}_asc_T"]  = r["asc_avg"]
        if s != 1.0:
            export[f"mu0M_{geom}_desc_scaled_T"] = r["desc_avg"] * s
            export[f"mu0M_{geom}_asc_scaled_T"]  = r["asc_avg"] * s
    pd.DataFrame(export).to_csv(
        os.path.join(batch_folder, f"Averaged_v2_{diam}um_{ts}.csv"), index=False)

    df_res.to_csv(os.path.join(batch_folder, f"Metrics_v2_{diam}um_{ts}.csv"),
                  index=False)

    print(f"\nDONE. All outputs saved in: {batch_folder}")


if __name__ == "__main__":
    main()